# Tutorial 5: Network Analysis - Co-Pressing Partnerships

This tutorial demonstrates advanced network analysis of pressing partnerships. You'll learn:

- How to compute pressing affinity metrics (Jaccard, cosine similarity)
- How to build and analyze pressing networks
- How to identify key pressers using centrality metrics
- How to detect pressing communities
- How to interpret network structure tactically

## Prerequisites

Complete Tutorials 1-3 to have extracted build-ups and trained models.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from networkx.algorithms import community

# Network analysis modules
from src.models.pressing_affinity import compute_pressing_affinity
from src.models.gmm_zones import identify_pressers
from src.models.config import GMMConfig
from src.analysis.network_centrality import (
    compute_centrality_metrics,
    identify_pressing_communities
)
from src.viz.network import build_pressing_network, plot_pressing_network
from src.features.services.window_loader import WindowLoader
from src.features.services.normalization import normalize_coordinates
from src.features.services.possession import infer_ball_carrier
from src.features.services.utils import prepare_frame_data, time_to_seconds
from src.features.services.metadata import enrich_with_team_id

# Configuration
PROCESSED_ROOT = Path("data/processed/rm_pressing_tutorial")
OUTPUT_DIR = Path("visualizations/networks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style("whitegrid")

## Part 1: Building the Pressing Network

### Step 1: Identify Active Pressers Across Build-Ups

Collect presser sets from all build-ups.

In [ ]:
from tqdm.notebook import tqdm

loader = WindowLoader(PROCESSED_ROOT)
index = loader.index
gmm_config = GMMConfig()

presser_sets = []
build_up_presser_map = {}  # Map build_up_id → pressers

print(f"Analyzing {len(index)} build-ups...")

for bid in tqdm(index['build_up_id'].tolist(), desc="Identifying pressers"):
    try:
        df = loader.load_build_up(bid)
        meta = loader.get_metadata(bid)
        
        df = prepare_frame_data(df)
        df = enrich_with_team_id(df, meta['game_id'])
        df_norm = normalize_coordinates(df, meta.get('gk_side', 'left'))
        df_norm = infer_ball_carrier(df_norm, meta.get('opponent_team_id'))
        df_norm['time_seconds'] = df_norm['time'].apply(time_to_seconds)
        
        pressers = identify_pressers(df_norm, gmm_config)
        if len(pressers) > 0:
            presser_sets.append(set(pressers))
            build_up_presser_map[bid] = pressers
    except Exception:
        continue

print(f"\nCollected {len(presser_sets)} presser sets")
print(f"Average pressers per build-up: {np.mean([len(s) for s in presser_sets]):.1f}")

# Unique players who pressed
all_pressers = set()
for s in presser_sets:
    all_pressers.update(s)

print(f"Total unique pressers: {len(all_pressers)}")

### Step 2: Compute Affinity Matrices

Calculate how often players press together using Jaccard and cosine similarity.

In [ ]:
# Compute both affinity metrics
affinity_jaccard = compute_pressing_affinity(presser_sets, metric='jaccard')
affinity_cosine = compute_pressing_affinity(presser_sets, metric='cosine')

print(f"Jaccard Affinity Matrix: {affinity_jaccard.shape}")
print(f"Cosine Affinity Matrix: {affinity_cosine.shape}")

# Compare metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Jaccard heatmap
sns.heatmap(affinity_jaccard, cmap='YlOrRd', cbar_kws={'label': 'Jaccard Similarity'},
           linewidths=0.5, ax=ax1, square=True)
ax1.set_title('Pressing Affinity - Jaccard Similarity', fontsize=14, fontweight='bold')
ax1.set_xlabel('Player ID')
ax1.set_ylabel('Player ID')

# Cosine heatmap
sns.heatmap(affinity_cosine, cmap='YlGnBu', cbar_kws={'label': 'Cosine Similarity'},
           linewidths=0.5, ax=ax2, square=True)
ax2.set_title('Pressing Affinity - Cosine Similarity', fontsize=14, fontweight='bold')
ax2.set_xlabel('Player ID')
ax2.set_ylabel('Player ID')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "affinity_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nInterpretation:")
print("  - Jaccard: Measures overlap in presence across build-ups")
print("  - Cosine: Measures co-occurrence pattern similarity")

### Step 3: Build Network Graph

Create network with edges weighted by affinity (using Jaccard).

In [ ]:
# Build network with threshold to filter weak connections
threshold = 0.2  # Only include edges with affinity >= 0.2

G = build_pressing_network(affinity_jaccard, threshold=threshold)

print(f"Pressing Network:")
print(f"  Nodes: {G.number_of_nodes()} (players)")
print(f"  Edges: {G.number_of_edges()} (partnerships)")
print(f"  Threshold: {threshold}")
print(f"  Density: {nx.density(G):.3f}")

if nx.is_connected(G):
    print(f"  Network is connected")
    print(f"  Diameter: {nx.diameter(G)}")
else:
    print(f"  Network is disconnected ({nx.number_connected_components(G)} components)")

## Part 2: Centrality Analysis

### Step 4: Compute Centrality Metrics

Identify key players based on:
- **Degree centrality**: Number of pressing partners
- **Betweenness centrality**: Bridges between player groups
- **Closeness centrality**: Proximity to all other pressers
- **PageRank**: Importance considering partner importance

In [ ]:
centrality_df = compute_centrality_metrics(G)

print("Centrality Metrics:")
print(centrality_df.head(10))

# Visualize centrality distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Degree centrality
axes[0, 0].bar(range(len(centrality_df)), centrality_df['degree_centrality'], 
              color='steelblue', edgecolor='black')
axes[0, 0].set_title('Degree Centrality', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Player Rank')
axes[0, 0].set_ylabel('Centrality')
axes[0, 0].grid(alpha=0.3, axis='y')

# Betweenness centrality
axes[0, 1].bar(range(len(centrality_df)), centrality_df['betweenness_centrality'], 
              color='coral', edgecolor='black')
axes[0, 1].set_title('Betweenness Centrality', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Player Rank')
axes[0, 1].set_ylabel('Centrality')
axes[0, 1].grid(alpha=0.3, axis='y')

# Closeness centrality
if 'closeness_centrality' in centrality_df.columns:
    axes[1, 0].bar(range(len(centrality_df)), centrality_df['closeness_centrality'], 
                  color='green', edgecolor='black')
    axes[1, 0].set_title('Closeness Centrality', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Player Rank')
    axes[1, 0].set_ylabel('Centrality')
    axes[1, 0].grid(alpha=0.3, axis='y')

# PageRank
axes[1, 1].bar(range(len(centrality_df)), centrality_df['pagerank'], 
              color='purple', edgecolor='black')
axes[1, 1].set_title('PageRank', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Player Rank')
axes[1, 1].set_ylabel('PageRank')
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "centrality_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

### Step 5: Identify Key Pressers

Rank players by centrality to find pressing leaders.

In [ ]:
print("Top 10 Pressers by Degree Centrality (Most Partnerships):")
print(centrality_df.nlargest(10, 'degree_centrality')[['degree_centrality', 'pagerank']])

print("\nTop 10 Pressers by Betweenness Centrality (Connectors):")
print(centrality_df.nlargest(10, 'betweenness_centrality')[['betweenness_centrality', 'pagerank']])

print("\nTop 10 Pressers by PageRank (Overall Importance):")
print(centrality_df.nlargest(10, 'pagerank')[['degree_centrality', 'betweenness_centrality', 'pagerank']])

# Tactical interpretation
top_degree = centrality_df.nlargest(1, 'degree_centrality').index[0]
top_between = centrality_df.nlargest(1, 'betweenness_centrality').index[0]
top_pagerank = centrality_df.nlargest(1, 'pagerank').index[0]

print(f"\n{'='*70}")
print("Tactical Insights:")
print(f"{'='*70}")
print(f"  Most Connected Presser: Player {top_degree}")
print(f"    → Presses with many different partners (versatile)")
print(f"\n  Most Critical Connector: Player {top_between}")
print(f"    → Bridges different pressing groups (coordinator)")
print(f"\n  Most Important Overall: Player {top_pagerank}")
print(f"    → High influence considering partner quality (leader)")

## Part 3: Community Detection

### Step 6: Detect Pressing Communities

Identify groups of players who frequently press together (e.g., left-side unit, right-side unit).

In [ ]:
communities = identify_pressing_communities(G)

print(f"Detected {len(communities)} pressing communities:\n")
for i, comm in enumerate(communities):
    print(f"Community {i}: {len(comm)} players")
    print(f"  Players: {sorted(list(comm))}")
    print()

# Add community assignments to centrality df
player_to_community = {}
for i, comm in enumerate(communities):
    for player in comm:
        player_to_community[player] = i

centrality_df['community'] = centrality_df.index.map(player_to_community)

### Step 7: Visualize Network with Communities

Plot network with nodes colored by community membership.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Layout
pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)

# Draw edges
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
nx.draw_networkx_edges(
    G, pos,
    width=[w * 5 for w in edge_weights],
    alpha=0.3,
    edge_color='gray',
    ax=ax
)

# Draw nodes colored by community
community_colors = plt.cm.Set3(range(len(communities)))
for i, comm in enumerate(communities):
    nx.draw_networkx_nodes(
        G, pos,
        nodelist=list(comm),
        node_size=800,
        node_color=[community_colors[i]],
        edgecolors='black',
        linewidths=2,
        label=f'Community {i}',
        ax=ax
    )

# Labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)

ax.set_title('Pressing Network with Communities', fontsize=16, fontweight='bold')
ax.legend(loc='upper right', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "network_communities.png", dpi=150, bbox_inches='tight')
plt.show()

print("Interpretation:")
print("  - Communities may represent positional units (e.g., forwards, midfielders, defenders)")
print("  - Dense intra-community edges = coordinated pressing within unit")
print("  - Inter-community edges = coordination across units")

## Part 4: Network Statistics

### Step 8: Global Network Properties

Analyze overall network structure.

In [ ]:
print(f"{'='*70}")
print("Network Statistics:")
print(f"{'='*70}")

print(f"\nBasic Properties:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.3f}")
print(f"    → {nx.density(G)*100:.1f}% of possible connections exist")

print(f"\nDegree Distribution:")
degrees = dict(G.degree())
degree_values = list(degrees.values())
print(f"  Mean degree: {np.mean(degree_values):.2f}")
print(f"  Median degree: {np.median(degree_values):.0f}")
print(f"  Max degree: {np.max(degree_values)}")

if nx.is_connected(G):
    print(f"\nConnectivity:")
    print(f"  Network is connected")
    print(f"  Average shortest path: {nx.average_shortest_path_length(G):.2f}")
    print(f"  Diameter: {nx.diameter(G)}")
else:
    print(f"\nConnectivity:")
    print(f"  Network has {nx.number_connected_components(G)} components")
    largest_cc = max(nx.connected_components(G), key=len)
    print(f"  Largest component: {len(largest_cc)} nodes")

print(f"\nClustering:")
clustering = nx.average_clustering(G, weight='weight')
print(f"  Average clustering coefficient: {clustering:.3f}")
print(f"    → Measures how interconnected player neighborhoods are")

print(f"\nCommunities:")
print(f"  Number of communities: {len(communities)}")
print(f"  Modularity: {community.modularity(G, communities):.3f}")
print(f"    → Higher modularity = stronger community structure")

### Step 9: Compare Centrality vs. Pressing Frequency

Do the most central players also press most often?

In [ ]:
# Count pressing frequency
presser_counts = {}
for pressers in presser_sets:
    for p in pressers:
        presser_counts[p] = presser_counts.get(p, 0) + 1

centrality_df['press_count'] = centrality_df.index.map(presser_counts)

# Scatter plot: PageRank vs. Press Count
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(
    centrality_df['pagerank'],
    centrality_df['press_count'],
    s=centrality_df['degree_centrality'] * 1000,  # Size by degree
    c=centrality_df['community'],
    cmap='Set3',
    alpha=0.7,
    edgecolor='black',
    linewidth=1.5
)

# Annotate top players
top_players = centrality_df.nlargest(5, 'pagerank')
for player_id, row in top_players.iterrows():
    ax.annotate(
        f'P{player_id}',
        (row['pagerank'], row['press_count']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=10,
        fontweight='bold'
    )

ax.set_xlabel('PageRank (Network Importance)', fontsize=12)
ax.set_ylabel('Press Count (Frequency)', fontsize=12)
ax.set_title('Network Centrality vs. Pressing Frequency\n(Size = Degree Centrality)', 
            fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.colorbar(scatter, ax=ax, label='Community', ticks=range(len(communities)))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "centrality_vs_frequency.png", dpi=150, bbox_inches='tight')
plt.show()

# Correlation
corr = centrality_df[['pagerank', 'press_count']].corr().iloc[0, 1]
print(f"\nCorrelation (PageRank vs. Press Count): {corr:.3f}")
if corr > 0.5:
    print("  → High correlation: Central players press more frequently")
elif corr < -0.5:
    print("  → Negative correlation: Central players are specialized (quality over quantity)")
else:
    print("  → Weak correlation: Centrality and frequency are independent")

## Part 5: Tactical Interpretation

### Step 10: Generate Network Summary Report

Synthesize findings into actionable insights.

In [ ]:
print(f"{'='*70}")
print("PRESSING NETWORK ANALYSIS SUMMARY")
print(f"{'='*70}")

print(f"\n1. NETWORK STRUCTURE")
print(f"   - {G.number_of_nodes()} active pressers identified")
print(f"   - {G.number_of_edges()} pressing partnerships (affinity ≥ {threshold})")
print(f"   - Network density: {nx.density(G):.1%}")
print(f"   - {len(communities)} distinct pressing communities detected")

print(f"\n2. KEY PRESSERS")
top3_pagerank = centrality_df.nlargest(3, 'pagerank')
for rank, (player_id, row) in enumerate(top3_pagerank.iterrows(), 1):
    print(f"   #{rank}. Player {player_id}:")
    print(f"       PageRank: {row['pagerank']:.3f}")
    print(f"       Partnerships: {int(row['degree_centrality'] * (G.number_of_nodes() - 1))}")
    print(f"       Press Count: {int(row['press_count'])} build-ups")

print(f"\n3. PRESSING COMMUNITIES")
for i, comm in enumerate(communities):
    print(f"   Community {i}: {len(comm)} players")
    # Average centrality of community
    comm_centrality = centrality_df.loc[list(comm), 'pagerank'].mean()
    print(f"     Average importance: {comm_centrality:.3f}")
    print(f"     Players: {sorted(list(comm))}")

print(f"\n4. STRONGEST PARTNERSHIPS")
edges = [(u, v, G[u][v]['weight']) for u, v in G.edges()]
top5_edges = sorted(edges, key=lambda x: x[2], reverse=True)[:5]
for rank, (u, v, w) in enumerate(top5_edges, 1):
    print(f"   #{rank}. Player {u} ↔ Player {v}: {w:.3f}")

print(f"\n5. TACTICAL INSIGHTS")
if len(communities) == 2:
    print(f"   - Two communities suggest left/right or front/back split")
elif len(communities) == 3:
    print(f"   - Three communities likely represent forwards, midfielders, defenders")
else:
    print(f"   - {len(communities)} communities indicate complex pressing coordination")

avg_clustering = nx.average_clustering(G, weight='weight')
if avg_clustering > 0.5:
    print(f"   - High clustering ({avg_clustering:.2f}) = tightly coordinated pressing units")
else:
    print(f"   - Moderate clustering ({avg_clustering:.2f}) = flexible pressing partnerships")

print(f"\n{'='*70}")

## Summary

You've learned how to:
1. ✅ Compute pressing affinity matrices (Jaccard & cosine)
2. ✅ Build weighted pressing networks
3. ✅ Calculate centrality metrics (degree, betweenness, closeness, PageRank)
4. ✅ Identify key pressers and connectors
5. ✅ Detect pressing communities using Louvain algorithm
6. ✅ Visualize networks with community coloring
7. ✅ Analyze global network properties
8. ✅ Generate tactical insights from network structure

## Key Takeaways

- **Network centrality** reveals pressing leaders and coordinators
- **Communities** expose positional units and tactical organization
- **Edge weights** quantify partnership strength
- **Network density** indicates overall coordination level
- **Clustering** measures local cohesion (sub-unit coordination)

## Applications

- **Player recruitment**: Identify players who fit existing pressing partnerships
- **Tactical adjustments**: Strengthen weak connections, reinforce strong communities
- **Match preparation**: Understand opponent's pressing network structure
- **Performance evaluation**: Track network evolution across seasons

## Next Steps

- Compare networks across different match contexts (home/away, opponent quality)
- Temporal analysis: How do networks evolve over a season?
- Predictive modeling: Can network properties predict pressing success?